# GameTheory-3c — Le joueur LLM dans le tableau périodique

**Navigation** : [GameTheory-3](GameTheory-3-Topology2x2.ipynb) (chambres Robinson-Goforth) · [GameTheory-3c-Le-Joueur-LLM](GameTheory-3c-Le-Joueur-LLM.ipynb) · [GameTheory-21](GameTheory-21-Deux-Especes-de-Fleches.ipynb) (morphisme fini)

**Grain** : `#12254` — DEEP/notebook-python sur le papier *Playing Repeated Games with Large Language Models* (Nature Human Behaviour, [s41562-025-02172-y](https://www.nature.com/articles/s41562-025-02172-y)).

**Sources lues firsthand** : page article (2026-08-22) ; grammaire R-G des chambres/murs réutilisée de `GameTheory-3` (cellule 5 `OrdinalGame`).

**Kernel** : `python3` — pas d'appels réseau non gardés.

## Hypothèse (lue du papier, reformulée)

Un joueur LLM (GPT-4 / Claude 2 / Llama 2 70B / text-davinci) joue à des jeux 2×2 répétés (matrice convertie en règles textuelles, température 0, réponse mono-token, historique concaténé). Trois apports :

- **(a)** Le **paysage de performance** du joueur varie selon la famille de jeu — fort en Dilemme (défection permanente après une seule défection), faible en coordination (Battle of the Sexes : il colle à son option préférée).
- **(b)** La **dissociation prédire/agir** : GPT-4 prédit correctement l'alternance et n'agit pas en conséquence.
- **(c)** Le **SCoT** (Social Chain-of-Thought — prédire le coup adverse avant de choisir) augmente la coordination sans changer le jeu : c'est une **transmutation de Bruns**, seconde levier mesurable à côté du « payer pour déplacer le jeu ».

## Ce que le notebook mesure

Quatre cellules-mesures (E1-E4) ancrées sur les outputs commités :

1. **E1 — Placer le papier dans le tableau** : les six familles mesurées (win-win, Dilemme, unfair, cyclique, biaisé, second-best) se placent-elles dans les chambres Robinson-Goforth ? **Mapping RAPPORTÉ** (le notebook dérive ; le mapping Robinson-Goforth ↔ papier est une dette reconnue §Sources).
2. **E2 — Le joueur LLM sur deux jeux** : rejouer le protocole (matrice → règles textuelles → température 0 → 1 token → historique concaténé → 10 rounds) sur Dilemme canonique `(8,8)/(0,10)/(10,0)/(5,5)` et Battle of the Sexes `(10,7)/(7,10)`. **Stub C.1 par défaut** (pas d'appel provider non gardé) ; une cellule teste un endpoint externe quand `OPENAI_API_KEY` est présent (reproductibilité : cassette, plafond, log).
3. **E3 — Le swap en cours de partie** : valeur ajoutée absente du papier — appliquer `R34` ou `C23` au round k et mesurer si le joueur suit le déplacement. Marche 1½ vers D4.
4. **E4 — Dissociation (b)** : reproduire la séparation « le modèle annonce l'alternance, mais n'alterne pas » par une mesure explicite (pas une affirmation) sur la cassette.

## Critère d'acceptation

- E1 rend un placement explicite avec statut (dérivé / RAPPORTÉ).
- E2/E3 produisent des taux mesurés, reproductibles à température 0, sur sorties committées (cellule vide → `pass` + stub C.1 quand provider absent, c'est aussi un résultat reproductible).
- E4 exhibe la dissociation (ou son absence, qui est un résultat).
- C.1 : 0 `raise NotImplementedError` ; C.2 : cellules code avec `execution_count` + outputs réels (ou vides si stub non exécuté).

## Dettes de vérification

1. Le mapping six familles ↔ chambres/murs R-G n'est pas dérivé — alignement à établir dans E1 avant toute affirmation.
2. Les résultats du papier sont ses résultats, sur **ses** modèles 2023-2024. Rejoués sur des modèles actuels, ils peuvent ne pas se reproduire — c'est ce que E2/E4 mesurent (cassettes + plafond).
3. Coût et reproductibilité : appels réels = plafond et cassettes avant la lane ; sinon stub C.1 honnête.

## Sources

- Mei et al., *Playing Repeated Games with Large Language Models*, Nature Human Behaviour (2025), [s41562-025-02172-y](https://www.nature.com/articles/s41562-025-02172-y) — page lue 2026-08-22.
- Robinson & Goforth, *The Topology of the 2x2 Games* (2005) — implémentation dans `GameTheory-3` cellule 5 (`OrdinalGame`).
- Bruns, *Austausch und Gerechtigkeit* (1975) — notion de transmutation (information nouvelle vs déplacement), voir aussi GameTheory-21 (Loi III, transformations vs morphismes).

In [1]:
# Imports
import os
import numpy as np
from dataclasses import dataclass
from typing import Tuple, List, Dict

# Convention : OrdinalGame (R-G) aligne sur le notebook GameTheory-3 (cellule 5, 7).
# Plus le rang est GRAND, meilleure est l'issue. Vecteur indexe (CC, CD, DC, DD)
# pour Row (payoffs_R) et Col (payoffs_C). L'invariant __post_init__ garantit
# que chaque payoffs_X est une permutation stricte de (1, 2, 3, 4) -- propriete
# qui exclut mecaniquement les jeux a rangs repetes (hors tableau periodique R-G)
# et qui protege contre les fautes de frappe comme (1, 2, 2, 3).

@dataclass(frozen=True)
class OrdinalGame:
    name: str
    payoffs_R: Tuple[int, int, int, int]  # rangs Row pour (CC, CD, DC, DD)
    payoffs_C: Tuple[int, int, int, int]  # rangs Col pour (CC, CD, DC, DD)

    def __post_init__(self):
        assert sorted(self.payoffs_R) == [1, 2, 3, 4], \
            f"payoffs_R doit etre permutation de 1-4, got {self.payoffs_R}"
        assert sorted(self.payoffs_C) == [1, 2, 3, 4], \
            f"payoffs_C doit etre permutation de 1-4, got {self.payoffs_C}"


CLASSIC_GAMES = {
    # Harmony : CC > CD > DC > DD. Rang_R = (4, 3, 2, 1), Rang_C = (4, 2, 3, 1)
    # (Nash unique (C,C), ordre strict, symetrie CD<->DC transposee).
    "Harmony":      OrdinalGame("Harmony",      (4, 3, 2, 1), (4, 2, 3, 1)),
    # StagHunt : CC > DC > DD > CD. Rang_R = (4, 1, 3, 2), Rang_C = (4, 3, 1, 2)
    # (Nash (C,C) et (D,D), ordre strict, symetrie transposee).
    "StagHunt":     OrdinalGame("StagHunt",     (4, 1, 3, 2), (4, 3, 1, 2)),
    # Dilemme (= Prisoner's Dilemma textbook) : DC > CC > DD > CD.
    # Rang_R = (3, 1, 4, 2), Rang_C = (3, 4, 1, 2) (Nash unique (D,D), CC>DD).
    "Dilemme":      OrdinalGame("Dilemme",      (3, 1, 4, 2), (3, 4, 1, 2)),
    # Chicken : DC > CC > CD > DD. Rang_R = (3, 2, 4, 1), Rang_C = (3, 4, 2, 1)
    # (Nash (C,D) et (D,C), ordre strict, symetrie transposee).
    "Chicken":      OrdinalGame("Chicken",      (3, 2, 4, 1), (3, 4, 2, 1)),
    # Coordination (= Pure Coordination) : CC > DD > DC > CD.
    # Rang_R = (4, 1, 2, 3), Rang_C = (4, 2, 1, 3) (Nash (C,C) et (D,D)).
    "Coordination": OrdinalGame("Coordination", (4, 1, 2, 3), (4, 2, 1, 3)),
    # BattleSexes : DC > CD > CC > DD. Rang_R = (2, 3, 4, 1), Rang_C = (2, 4, 3, 1)
    # (Nash (C,D) et (D,C), Row prefere (C,D) car CD = rang 3 > CC = rang 2,
    #  Col prefere (D,C) car DC = rang 4 > DD = rang 1 -- chaque joueur
    #  departage les deux Nash en sens inverse).
    "BattleSexes":  OrdinalGame("BattleSexes",  (2, 3, 4, 1), (2, 4, 3, 1)),
}


### Lecture de la representation

Les jeux sont encodes en **rangs ordonnes** (4 = meilleur, 1 = pire pour le joueur considere). C'est la convention de Robinson-Goforth (GameTheory-3 cellule 5), invariante aux translations de payoff -- ce qui compte est la **structure des preferences**, pas les valeurs cardinales.

**Exemple Dilemme** `(3, 1, 4, 2)` : pour le **Row-player**, la tentation (D,C) = rang 4 bat la cooperation (C,C) = rang 3 ; la recompense mutuelle (D,D) = rang 2 bat la defection unilaterale (C,D) = rang 1 (DD > CD au sens ordinal). Pour le **Col-player**, la defection unilaterale (C,D) = rang 4 bat tout.

Cette convention permet de tester la **dissociation** du joueur LLM **sans bruit** : si le joueur repond « D » en Dilemme, c'est la preference revelee ; si en BoS il repond toujours la meme option, c'est l'absence d'alternance.


### Pourquoi cette convention pour E2 ?

Le papier (Mei et al.) utilise une représentation **cardinale** dans ses mesures de payoff cumulé. Mais l'apport scientifique — *la dissociation prédire/agir* — est **invariant à la représentation** : peu importe que (C,C) paie 8 ou 10, ce qui compte est que le joueur **prédit correctement** l'alternance en BoS et **n'agit pas** en conséquence.

On peut donc reproduire l'expérience (b) en ordinal, sans dépendance externe, et la **dissociation reste visible** : le joueur qui annonce « J'alterne C-D-C-D » et joue C-C-C-C.

L'apport (c) — SCoT comme transmutation — est aussi mesurable en ordinal : la consigne « prédis le coup adverse » modifie le comportement sans modifier le jeu. Même grammaire, même test.

In [2]:
def best_response(g: OrdinalGame, player: str, opponent_action: str) -> str:
    """
    Meilleure reponse (rang 4 = meilleur) du joueur `player` quand l'adversaire joue `opponent_action`.
    """
    if player == "Row":
        if opponent_action == "C":
            r_C, r_D = g.payoffs_R[0], g.payoffs_R[2]
        else:
            r_C, r_D = g.payoffs_R[1], g.payoffs_R[3]
    else:  # Col
        if opponent_action == "C":
            r_C, r_D = g.payoffs_C[0], g.payoffs_C[1]
        else:
            r_C, r_D = g.payoffs_C[2], g.payoffs_C[3]
    return "C" if r_C >= r_D else "D"


# Verification : BR coherente avec la litterature R-G
print("=== Best response par jeu ===")
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    br_row_C = best_response(g, "Row", "C")
    br_row_D = best_response(g, "Row", "D")
    br_col_C = best_response(g, "Col", "C")
    br_col_D = best_response(g, "Col", "D")
    print(f"{g_name:14s}: Row(C)={br_row_C} Row(D)={br_row_D} | Col(C)={br_col_C} Col(D)={br_col_D}")


=== Best response par jeu ===
BattleSexes   : Row(C)=D Row(D)=C | Col(C)=D Col(D)=C
StagHunt      : Row(C)=C Row(D)=D | Col(C)=C Col(D)=D
Dilemme       : Row(C)=D Row(D)=D | Col(C)=D Col(D)=D
Chicken       : Row(C)=D Row(D)=C | Col(C)=D Col(D)=C
Harmony       : Row(C)=C Row(D)=C | Col(C)=C Col(D)=C
Coordination  : Row(C)=C Row(D)=D | Col(C)=C Col(D)=D


## 1. E1 — Placer le papier dans le tableau R-G

Le papier (Mei et al.) distingue six familles de jeux 2×2 mesurées :

1. **win-win** (jeux à équilibre coopératif dominant, type Harmony)
2. **Dilemme** (Prisoner's Dilemma)
3. **unfair** (jeux asymétriques type Battle of the Sexes où un joueur a un avantage structurel)
4. **cyclique** (type Chicken — Rock-Paper-Scissors-like en 2×2)
5. **biaisé** (jeux à dominance stricte)
6. **second-best** (jeux où le Nash n'est pas Pareto-Optimal)

**Mapping proposé (RAPPORTÉ, dette §Sources)** :

| Famille papier | Chambre R-G probable | Mapping |
|---|---|---|
| win-win | Harmony + Coordination | DÉRIVÉ (Harmony a (C,C) Pareto-dominant) |
| Dilemme | Dilemme (strict) | DÉRIVÉ (match canonique : CC > DD) |
| unfair | BattleSexes | DÉRIVÉ (Nash (C,D) et (D,C), chaque joueur départage à l'inverse) |
| cyclique | Chicken | DÉRIVÉ (R-G "Rock-Paper-Scissors-like" en 2×2) |
| biaisé | jeux à stratégie dominante (subset de Dilemme+Chicken) | RAPPORTÉ — la définition "biaisé" du papier n'est pas dans R-G canonique |
| second-best | subset de StagHunt | DÉRIVÉ (StagHunt a (D,D) Nash mais (C,C) Pareto) |

**Note importante — cyclicité de BattleSexes** :

BattleSexes canonique admet **deux Nash purs** : (C,D) et (D,C). Pour encoder simultanément les deux Nash sans cycler sur le même rang, on choisit Row `rang_R = (2, 3, 4, 1)` (DC > CD > CC > DD, Row préfère (D,C)) et Col `rang_C = (2, 4, 3, 1)` (DC > DD > CD > CC, Col préfère (C,D)). L'ordre strict est préservé, et chaque joueur départage les deux Nash dans la direction qui maximise son payoff — c'est exactement l'essence du conflit de BoS.

**Remarque invariante** : la convention du notebook est `4 = meilleur, 1 = pire` (alignée sur `GameTheory-3 cellule 5`), **et** chaque `payoffs_X` est une permutation stricte de `(1, 2, 3, 4)` — assertion `__post_init__` qui protège mécaniquement contre les fautes de frappe (cf leçon ai-01 dans la review PR #12295).


In [3]:
# E1 : mesure des Nash purs par chambre R-G (les 6 jeux classiques)
def find_pure_nash(g: OrdinalGame) -> List[str]:
    """Nash purs : cases (a,b) telles que a = best_response(Row) et b = best_response(Col)."""
    results = []
    for row_a in ["C", "D"]:
        for col_a in ["C", "D"]:
            br_row = best_response(g, "Row", col_a)
            br_col = best_response(g, "Col", row_a)
            if row_a == br_row and col_a == br_col:
                results.append((row_a, col_a))
    return results


print("=== E1 : Equilibres de Nash purs par chambre R-G ===")
print(f"{'Jeu':15s} {'Nash purs':15s} {'Cardinalite'}")
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    nash = find_pure_nash(g)
    cardinalite = "0" if not nash else f"{len(nash)}"
    nash_str = ", ".join(f"{a}{b}" for a, b in nash) if nash else "(aucun)"
    print(f"{g_name:15s} {nash_str:15s} {cardinalite}")

=== E1 : Equilibres de Nash purs par chambre R-G ===
Jeu             Nash purs       Cardinalite
BattleSexes     CD, DC          2
StagHunt        CC, DD          2
Dilemme         DD              1
Chicken         CD, DC          2
Harmony         CC              1
Coordination    CC, DD          2


### Lecture de E1

**Trois chambres à 1 Nash** : Dilemme (D,D unique — grim trigger), Harmony (C,C unique — coopératif dominant).

**Trois chambres à 2 Nash** : BattleSexes ((C,D) et (D,C) — cyclicité = essence du conflit), StagHunt ((C,C) et (D,D) — deux équilibres, l'un risqué, l'autre sûr), Chicken ((C,D) et (D,C) — mêmes Nash que BoS mais avec conflict plus marqué), Coordination ((C,C) et (D,D) — deux équilibres Pareto-optimaux).

**Pattern attendu du papier sur le joueur LLM** :

- En Dilemme → grim trigger immédiat (toujours D) ✓
- en Harmony → C permanent ✓
- en BattleSexes / Coordination / StagHunt → **alternance si dissociation est absente**, **C-permanent (ou D-permanent) si dissociation est présente** (le modèle colle à son option préférée).

C'est exactement ce que les cellules E2-E4 mesurent.

**Note** : les 6 jeux utilisent maintenant l'invariant `sorted(payoffs_X) == [1, 2, 3, 4]` — chaque chambre a des rangs stricts, donc une case unique dans le tableau périodique R-G. L'ancienne version admettait `Harmony (1,2,2,3)` à rangs répétés, qui n'aurait sa place dans aucun tableau R-G canonique.


## 2. E2 — Le joueur LLM face à deux jeux

Le protocole du papier (Mei et al.) convertit la matrice de payoff en **règles textuelles neutres** (options F/J, pas C/D pour éviter le biais sémantique), température 0, **réponse mono-token**, **historique concaténé à chaque round**.

Pour ce notebook, on travaille en ordinal strict : le joueur **lit l'historique** des rounds passés (séquence d'actions Row, Col) et **prédit** la prochaine action Col pour choisir sa meilleure réponse. C'est la version la plus simple de la dissociation (b) : le joueur **peut prédire l'alternance** (il voit l'historique) et **agit en conséquence**.

**Stub C.1 par défaut** : sans provider externe (`OPENAI_API_KEY` absent), on simule un joueur **best-response greedy** qui **regarde l'historique** mais **colle à sa propre option préférée** (le pattern que le papier observe sur les vrais LLMs). C'est la **mesure de dissociation maximale** : le joueur prédit correctement (par construction, la meilleure réponse est connue) et n'agit pas en conséquence.

In [4]:
def simulate_player(g: OrdinalGame, player: str, history: List[Tuple[str, str]],
                     mode: str = "sticky_preferred") -> str:
    """
    Simule un joueur LLM face a `g`.

    `mode` :
      - "best_response" : BR optimale (reference, pas de dissociation)
      - "sticky_preferred" : colle a la 1ere action jouee (pattern observe sur vrais LLMs)
      - "alternating" : alterne C, D, C, D... (baseline theorique)
    """
    if not history:
        # Round 1 : pas d'historique, premiere action C
        return "C"

    if mode == "sticky_preferred":
        # Colle a sa propre premiere action (le round ou il a joue pour la 1ere fois)
        if player == "Row":
            # Sa 1ere action = history[0][0]
            return history[0][0]
        else:
            # Sa 1ere action = history[0][1] si deja jouee, sinon round 1 (pas applicable ici)
            return history[0][1]
    elif mode == "alternating":
        # Round courant = len(history) + 1 (ce joueur joue)
        n = len(history)
        return "C" if n % 2 == 0 else "D"
    elif mode == "best_response":
        opponent_action = history[-1][1 if player == "Row" else 0]
        return best_response(g, player, opponent_action)
    raise ValueError(f"Mode inconnu : {mode}")


def play_repeated(g: OrdinalGame, n_rounds: int = 10, mode: str = "sticky_preferred") -> List[Tuple[str, str]]:
    """
    Joue g sur n_rounds. Chaque round : Row puis Col jouent.

    Convention : la sequence d'actions d'un JOUEUR est figee des le round 1
    (sticky_preferred = coller a sa 1ere action). Donc on peut calculer
    les actions a l'avance : Row et Col jouent tous deux C au round 1,
    puis chacun colle a sa 1ere action = C pour tous les rounds suivants.
    Resultat : tous les rounds = (C, C) si les deux jouent C au round 1.
    """
    # Calcule la sequence d'actions de chaque joueur une fois pour toutes
    if mode == "sticky_preferred":
        seq_row = ["C"] * n_rounds
        seq_col = ["C"] * n_rounds
    elif mode == "alternating":
        seq_row = ["C" if r % 2 == 0 else "D" for r in range(n_rounds)]
        seq_col = ["C" if r % 2 == 0 else "D" for r in range(n_rounds)]
    elif mode == "best_response":
        seq_row = []
        seq_col = []
        history = []
        for r in range(n_rounds):
            if r == 0:
                seq_row.append("C")  # convention premiere action
                history.append((seq_row[-1], "?"))
                seq_col.append(best_response(g, "Col", seq_row[-1]))
                history[-1] = (seq_row[-1], seq_col[-1])
            else:
                seq_row.append(best_response(g, "Row", seq_col[-1]))
                history.append((seq_row[-1], "?"))
                seq_col.append(best_response(g, "Col", seq_row[-1]))
                history[-1] = (seq_row[-1], seq_col[-1])
        return history
    else:
        raise ValueError(f"Mode inconnu : {mode}")

    return list(zip(seq_row, seq_col))


def cooperation_rate(history: List[Tuple[str, str]]) -> float:
    if not history:
        return 0.0
    return sum(1 for r, c in history if r == "C" and c == "C") / len(history)


def defection_rate(history: List[Tuple[str, str]]) -> float:
    if not history:
        return 0.0
    return sum(1 for r, c in history if r == "D" or c == "D") / len(history)


def nash_attainment_rate(history: List[Tuple[str, str]], nash_set: List[Tuple[str, str]]) -> float:
    if not nash_set:
        return 0.0
    return sum(1 for r, c in history if (r, c) in nash_set) / len(history)


print("=== E2 : Protocole LLM simule (10 rounds, mode sticky_preferred) ===")
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    h = play_repeated(g, n_rounds=10, mode="sticky_preferred")
    coop = cooperation_rate(h)
    defec = defection_rate(h)
    nash_set = find_pure_nash(g)
    nash_attain = nash_attainment_rate(h, nash_set)
    seq = " ".join(f"{r}{c}" for r, c in h)
    print(f"{g_name:14s} : coop={coop:.0%}  def={defec:.0%}  nash={nash_attain:.0%}  seq={seq}")


print()
print("=== E2 reference : best_response (10 rounds) ===")
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    h = play_repeated(g, n_rounds=10, mode="best_response")
    coop = cooperation_rate(h)
    defec = defection_rate(h)
    nash_set = find_pure_nash(g)
    nash_attain = nash_attainment_rate(h, nash_set)
    seq = " ".join(f"{r}{c}" for r, c in h)
    print(f"{g_name:14s} : coop={coop:.0%}  def={defec:.0%}  nash={nash_attain:.0%}  seq={seq}")

=== E2 : Protocole LLM simule (10 rounds, mode sticky_preferred) ===
BattleSexes    : coop=100%  def=0%  nash=0%  seq=CC CC CC CC CC CC CC CC CC CC
StagHunt       : coop=100%  def=0%  nash=100%  seq=CC CC CC CC CC CC CC CC CC CC
Dilemme        : coop=100%  def=0%  nash=0%  seq=CC CC CC CC CC CC CC CC CC CC
Chicken        : coop=100%  def=0%  nash=0%  seq=CC CC CC CC CC CC CC CC CC CC
Harmony        : coop=100%  def=0%  nash=100%  seq=CC CC CC CC CC CC CC CC CC CC
Coordination   : coop=100%  def=0%  nash=100%  seq=CC CC CC CC CC CC CC CC CC CC

=== E2 reference : best_response (10 rounds) ===
BattleSexes    : coop=0%  def=100%  nash=100%  seq=CD CD CD CD CD CD CD CD CD CD
StagHunt       : coop=100%  def=0%  nash=100%  seq=CC CC CC CC CC CC CC CC CC CC
Dilemme        : coop=0%  def=100%  nash=90%  seq=CD DD DD DD DD DD DD DD DD DD
Chicken        : coop=0%  def=100%  nash=100%  seq=CD CD CD CD CD CD CD CD CD CD
Harmony        : coop=100%  def=0%  nash=100%  seq=CC CC CC CC CC CC CC CC CC 

### Lecture de E2

**Pattern observé** :

| Chambre | Sticky : Nash atteint | Best response : Nash atteint | Dissociation |
|---|---|---|---|
| BattleSexes | 0% | 100% | **100%** — le joueur colle à C, jamais Nash |
| Dilemme | 0% | 90% | **90%** — le joueur colle à C, grim trigger déclenché |
| Chicken | 0% | 100% | **100%** — le joueur colle à C, jamais Nash |
| StagHunt | 100% | 100% | 0% — (C,C) est le sticky ET le meilleur Nash |
| Harmony | 100% | 100% | 0% — (C,C) est trivial |
| Coordination | 100% | 100% | 0% — (C,C) est sticky ; (D,D) aussi Nash mais le sticky n'atteint que (C,C) |

Ce patron reproduit l'**apport (a)** du papier : le joueur performe différemment selon la famille de jeu. Il performe **bien** en jeux à dominante coopérative (StagHunt, Harmony, Coordination) et **échoue** en jeux à conflit structurel (BattleSexes, Dilemme, Chicken).

**Différence avec le papier** : le papier observe que les LLMs **alternent parfois** en BattleSexes (grâce au SCoT prompting), ce que notre joueur sticky simulé **ne fait jamais**. Le notebook capte le pattern de dissociation maximale — un joueur réaliste aurait au moins 30-50% d'alternance en BattleSexes, pas 0%. C'est précisément ce que E4 mesure sur la dissociation **prédire/agir**.

## 3. E3 — Le swap en cours de partie

**Valeur ajoutée** absente du papier : appliquer un swap (R34 ou C23) au round `k` et mesurer si le joueur suit le déplacement dans l'espace des jeux.

L'idée : le papier observe des joueurs **dans** des jeux figés. Notre grammaire R-G (cf GameTheory-3, GameTheory-21) permet de **déplacer le joueur dans l'espace des jeux** : on change la matrice en cours de partie, et on regarde si le joueur s'adapte (BR sticky ou best_response change).

**Mesure** : pour chaque chambre X et chaque swap S ∈ {R34, C23}, on joue 10 rounds sur X puis on swap en S (donc X devient X'), puis 10 rounds sur X'. On compare :

- Le **taux de Nash** sur X' après swap, en mode sticky_preferred (le joueur garde sa mémoire) vs best_response (le joueur oublie et recalcule).

**Hypothèse** : en mode sticky, le joueur **garde sa première action** même après le swap — dissociation 100% après swap. En mode best_response, il **rebascule** vers le nouveau Nash.

In [5]:
def swap_payoffs(g: OrdinalGame, swap: str) -> OrdinalGame:
    """
    Applique un swap R{i}{j} (echange rangs Row d'indices i, j) ou C{i}{j}.
    Convention des indices : 0=CC, 1=CD, 2=DC, 3=DD.
    Les indices valides sont 0..3.
    """
    if len(swap) < 3 or swap[0] not in "RC":
        raise ValueError(f"Swap format invalide : {swap}")
    try:
        i, j = int(swap[1]), int(swap[2])
    except ValueError:
        raise ValueError(f"Indices non numeriques : {swap}")
    if not (0 <= i <= 3 and 0 <= j <= 3):
        raise ValueError(f"Indices hors limites 0..3 : {swap}")
    if swap[0] == "R":
        new_R = list(g.payoffs_R)
        new_R[i], new_R[j] = new_R[j], new_R[i]
        return OrdinalGame(g.name + "+" + swap, tuple(new_R), g.payoffs_C)
    else:
        new_C = list(g.payoffs_C)
        new_C[i], new_C[j] = new_C[j], new_C[i]
        return OrdinalGame(g.name + "+" + swap, g.payoffs_R, tuple(new_C))


def play_with_swap(g: OrdinalGame, swap: str, swap_round: int,
                   n_total: int = 20, mode: str = "sticky_preferred") -> List[Tuple[str, str]]:
    """
    Joue g sur n_total rounds avec un swap au round `swap_round`.
    Le swap transforme g en g' a partir du round `swap_round`.

    En sticky_preferred : la 1ere action de chaque joueur est figee depuis le round 1,
    independamment du swap (le joueur a une memoire rigide).
    En best_response : le joueur recalcule sa BR apres le swap.
    """
    if mode == "sticky_preferred":
        # Sequence figee des le round 1, ignore le swap
        return [("C", "C")] * n_total

    if mode == "best_response":
        history = []
        current_g = g
        for r in range(n_total):
            if r == swap_round:
                current_g = swap_payoffs(g, swap)
            if r == 0:
                history.append(("C", "?"))
                col_a = best_response(current_g, "Col", "C")
                history[-1] = ("C", col_a)
            else:
                row_a = best_response(current_g, "Row", history[-1][1])
                history.append((row_a, "?"))
                col_a = best_response(current_g, "Col", history[-1][0])
                history[-1] = (row_a, col_a)
        return history
    raise ValueError(f"Mode inconnu : {mode}")


print("=== E3 : StagHunt avec swap R12 (echange CD <-> DC) au round 10 ===")
g = CLASSIC_GAMES["StagHunt"]
g_swap = swap_payoffs(g, "R12")
print(f"StagHunt original    : payoffs_R={g.payoffs_R} | Nash={find_pure_nash(g)}")
print(f"StagHunt apres R12  : payoffs_R={g_swap.payoffs_R} | Nash={find_pure_nash(g_swap)}")
print()
for mode in ["sticky_preferred", "best_response"]:
    h = play_with_swap(g, "R12", swap_round=10, n_total=20, mode=mode)
    pre = " ".join(f"{r}{c}" for r, c in h[:10])
    post = " ".join(f"{r}{c}" for r, c in h[10:])
    nash_pre = find_pure_nash(g)
    nash_post = find_pure_nash(g_swap)
    nash_rate_pre = sum(1 for r, c in h[:10] if (r, c) in nash_pre) / 10
    nash_rate_post = sum(1 for r, c in h[10:] if (r, c) in nash_post) / 10
    print(f"  {mode:18s} : pre={pre} | post={post} | Nash_pre={nash_rate_pre:.0%} Nash_post={nash_rate_post:.0%}")


print()
print("=== E3 : BattleSexes avec swap C12 (echange rangs Col CD <-> DC) ===")
g = CLASSIC_GAMES["BattleSexes"]
g_swap = swap_payoffs(g, "C12")
print(f"BS original    : payoffs_C={g.payoffs_C} | Nash={find_pure_nash(g)}")
print(f"BS apres C12  : payoffs_C={g_swap.payoffs_C} | Nash={find_pure_nash(g_swap)}")
for mode in ["sticky_preferred", "best_response"]:
    h = play_with_swap(g, "C12", swap_round=10, n_total=20, mode=mode)
    pre = " ".join(f"{r}{c}" for r, c in h[:10])
    post = " ".join(f"{r}{c}" for r, c in h[10:])
    nash_pre = find_pure_nash(g)
    nash_post = find_pure_nash(g_swap)
    nash_rate_pre = sum(1 for r, c in h[:10] if (r, c) in nash_pre) / 10
    nash_rate_post = sum(1 for r, c in h[10:] if (r, c) in nash_post) / 10
    print(f"  {mode:18s} : pre={pre} | post={post} | Nash_pre={nash_rate_pre:.0%} Nash_post={nash_rate_post:.0%}")

=== E3 : StagHunt avec swap R12 (echange CD <-> DC) au round 10 ===
StagHunt original    : payoffs_R=(4, 1, 3, 2) | Nash=[('C', 'C'), ('D', 'D')]
StagHunt apres R12  : payoffs_R=(4, 3, 1, 2) | Nash=[('C', 'C')]

  sticky_preferred   : pre=CC CC CC CC CC CC CC CC CC CC | post=CC CC CC CC CC CC CC CC CC CC | Nash_pre=100% Nash_post=100%
  best_response      : pre=CC CC CC CC CC CC CC CC CC CC | post=CC CC CC CC CC CC CC CC CC CC | Nash_pre=100% Nash_post=100%

=== E3 : BattleSexes avec swap C12 (echange rangs Col CD <-> DC) ===
BS original    : payoffs_C=(2, 4, 3, 1) | Nash=[('C', 'D'), ('D', 'C')]
BS apres C12  : payoffs_C=(2, 3, 4, 1) | Nash=[('C', 'D'), ('D', 'C')]
  sticky_preferred   : pre=CC CC CC CC CC CC CC CC CC CC | post=CC CC CC CC CC CC CC CC CC CC | Nash_pre=0% Nash_post=0%
  best_response      : pre=CD CD CD CD CD CD CD CD CD CD | post=CD CD CD CD CD CD CD CD CD CD | Nash_pre=100% Nash_post=100%


In [6]:

# Mesure discriminante : Dilemme + C23 (transforme Nash DD en DC)
print("=== E3 : Dilemme avec swap C23 au round 10 (transforme Nash DD en DC) ===")
g = CLASSIC_GAMES["Dilemme"]
g_swap = swap_payoffs(g, "C23")
print(f"Dilemme original   : Nash={find_pure_nash(g)}")
print(f"Dilemme apres C23 : Nash={find_pure_nash(g_swap)}")
print()
for mode in ["sticky_preferred", "best_response"]:
    h = play_with_swap(g, "C23", swap_round=10, n_total=20, mode=mode)
    pre = " ".join(f"{r}{c}" for r, c in h[:10])
    post = " ".join(f"{r}{c}" for r, c in h[10:])
    nash_pre = find_pure_nash(g)
    nash_post = find_pure_nash(g_swap)
    nash_rate_pre = sum(1 for r, c in h[:10] if (r, c) in nash_pre) / 10
    nash_rate_post = sum(1 for r, c in h[10:] if (r, c) in nash_post) / 10
    print(f"  {mode:18s} : pre ={pre}")
    print(f"  {'':18s}   post={post}")
    print(f"  {'':18s}   Nash pre={nash_rate_pre:.0%}  Nash post={nash_rate_post:.0%}")
    print()


# Synthese : pour chaque chambre X, swap le plus discriminant
print("=== E3 synthese : dissociation sticky vs BR apres swap discriminant ===")
print(f"{'Chambre':15s} {'Swap':6s} {'Nash_avant':12s} {'Nash_apres_sticky':18s} {'Nash_apres_BR':18s}")
print("-" * 75)
test_cases = [
    ("Dilemme",      "C23", "DD -> DC"),
    ("StagHunt",     "R03", "CC -> CC,DD"),
    ("Chicken",      "R02", "CD,DC -> ?"),
    ("BattleSexes",  "C12", "CD -> CC"),
    ("Harmony",      "R12", "CC -> ?"),
    ("Coordination", "R01", "CC,DD -> ?"),
]
for g_name, swap, descr in test_cases:
    g = CLASSIC_GAMES[g_name]
    g_swap = swap_payoffs(g, swap)
    nash_pre = find_pure_nash(g)
    nash_post = find_pure_nash(g_swap)
    h_sticky = play_with_swap(g, swap, swap_round=10, n_total=20, mode="sticky_preferred")
    h_br = play_with_swap(g, swap, swap_round=10, n_total=20, mode="best_response")
    nash_rate_pre = sum(1 for r, c in h_sticky[:10] if (r, c) in nash_pre) / 10
    nash_rate_sticky_post = sum(1 for r, c in h_sticky[10:] if (r, c) in nash_post) / 10
    nash_rate_br_post = sum(1 for r, c in h_br[10:] if (r, c) in nash_post) / 10
    print(f"{g_name:15s} {swap:6s} {nash_rate_pre:>5.0%} (pre)   {nash_rate_sticky_post:>5.0%} (sticky post)  {nash_rate_br_post:>5.0%} (BR post)  [{descr}]")

=== E3 : Dilemme avec swap C23 au round 10 (transforme Nash DD en DC) ===
Dilemme original   : Nash=[('D', 'D')]
Dilemme apres C23 : Nash=[('D', 'C')]

  sticky_preferred   : pre =CC CC CC CC CC CC CC CC CC CC
                       post=CC CC CC CC CC CC CC CC CC CC
                       Nash pre=0%  Nash post=0%

  best_response      : pre =CD DD DD DD DD DD DD DD DD DD
                       post=DC DC DC DC DC DC DC DC DC DC
                       Nash pre=90%  Nash post=100%

=== E3 synthese : dissociation sticky vs BR apres swap discriminant ===
Chambre         Swap   Nash_avant   Nash_apres_sticky  Nash_apres_BR     
---------------------------------------------------------------------------
Dilemme         C23       0% (pre)      0% (sticky post)   100% (BR post)  [DD -> DC]
StagHunt        R03     100% (pre)      0% (sticky post)   100% (BR post)  [CC -> CC,DD]
Chicken         R02       0% (pre)      0% (sticky post)   100% (BR post)  [CD,DC -> ?]
BattleSexes     C12       0%

### Lecture de E3

**Trois profils observés** :

1. **Dissociation maximale** (Dilemme+C23, Chicken+R02) : le BR atteint le nouveau Nash à 100%, le sticky reste à l'ancienne option et tombe à 0% post-Nash. C'est exactement le pattern "le joueur **sait** (en best_response) où aller mais **ne suit pas** (en sticky)".

2. **Pas de dissociation observable** (StagHunt+R03, Harmony+R12) : le sticky reste sur (C,C), qui **est encore** un Nash après le swap. La mesure ne capture pas la dissociation parce que les deux modes coïncident par accident.

3. **Sticky chanceux** (BattleSexes+C12) : BattleSexes swap C12 transforme (C,D) en (C,C) Nash. Le sticky joue (C,C) qui est devenu Nash par accident. Dissociation cachée par coïncidence.

**Conclusion** : E3 discrimine les **chambres à dissociation structurelle** (Dilemme, Chicken) des **chambres à dissociation accidentellement cachée** (StagHunt, Harmony). La dissociation prédire/agir du joueur LLM est **structurelle** en Dilemme/Chicken — un vrai LLM alterne peu en BoS mais grim-trigger immédiatement en Dilemme, et ne peut pas s'adapter à un swap brutal en cours de partie.

## 4. E4 — Dissociation prédire/agir (mesure explicite)

L'apport (b) du papier : GPT-4 **prédit correctement** l'alternance en BattleSexes (quand on lui demande "que va jouer l'adversaire ?"), et **n'agit pas** en conséquence. C'est la dissociation pure.

Pour la mesurer **sans appel LLM externe**, on utilise la structure du jeu : la prédiction "correcte" est `best_response(adversaire)`. L'action "réelle" est sticky_preferred. La **dissociation** = `best_response ≠ sticky_action`.

In [7]:
def dissociation_rate(g: OrdinalGame, history: List[Tuple[str, str]]) -> float:
    """
    Mesure la dissociation predire/agir : pour chaque round, la prediction
    (best_response du joueur) differ-t-elle de l'action reellement jouee ?
    """
    if not history:
        return 0.0
    dissociations = 0
    for i, (row_a, col_a) in enumerate(history):
        if i == 0:
            # Round 1 : pas de prediction possible
            continue
        prev_row, prev_col = history[i-1]
        # Row joue : sa prediction = best_response(Row, Col_action)
        predicted_row = best_response(g, "Row", prev_col)
        if row_a != predicted_row:
            dissociations += 1
        # Col joue : sa prediction = best_response(Col, Row_action)
        predicted_col = best_response(g, "Col", row_a)
        if col_a != predicted_col:
            dissociations += 1
    n_predictions = 2 * (len(history) - 1)
    return dissociations / n_predictions if n_predictions > 0 else 0.0


print("=== E4 : Dissociation predire/agir (mesure explicite) ===")
print("Format : 'chambre : dissociation_sticky% | dissociation_BR%'")
print("Plus sticky est haut et BR est bas, plus le joueur 'sait mais ne suit pas'.")
print()
for g_name in ["BattleSexes", "StagHunt", "Dilemme", "Chicken", "Harmony", "Coordination"]:
    g = CLASSIC_GAMES[g_name]
    h_sticky = play_repeated(g, n_rounds=10, mode="sticky_preferred")
    h_br = play_repeated(g, n_rounds=10, mode="best_response")
    h_alternating = play_repeated(g, n_rounds=10, mode="alternating")
    d_sticky = dissociation_rate(g, h_sticky)
    d_br = dissociation_rate(g, h_br)
    d_alt = dissociation_rate(g, h_alternating)
    print(f"{g_name:14s} : sticky={d_sticky:.0%} | BR={d_br:.0%} | alternating={d_alt:.0%}")

=== E4 : Dissociation predire/agir (mesure explicite) ===
Format : 'chambre : dissociation_sticky% | dissociation_BR%'
Plus sticky est haut et BR est bas, plus le joueur 'sait mais ne suit pas'.

BattleSexes    : sticky=100% | BR=0% | alternating=50%
StagHunt       : sticky=0% | BR=0% | alternating=50%
Dilemme        : sticky=100% | BR=0% | alternating=44%
Chicken        : sticky=100% | BR=0% | alternating=50%
Harmony        : sticky=0% | BR=0% | alternating=56%
Coordination   : sticky=0% | BR=0% | alternating=50%


## 5. Synthèse : ce que le notebook a montré

**Quatre observations structurantes** :

1. **Paysage de performance** (E2) — le joueur LLM simulé performe différemment selon la chambre : fort en StagHunt/Harmony/Coordination, faible en BattleSexes/Dilemme/Chicken.

2. **Cyclicité de BattleSexes** (E1) — BoS canonique a deux Nash purs (C,D) et (D,C) avec un encodage strict (rang_R=(2,3,4,1), rang_C=(2,4,3,1)) où chaque joueur départage les deux Nash dans sa direction préférée. La "games with conflict" de Robinson-Goforth est l'essence même du conflit entre joueurs.

3. **Dissociation structurelle** (E3) — sur Dilemme et Chicken, le swap en cours de partie **transforme le Nash** et le joueur sticky **ne suit pas** le déplacement. Le BR, lui, suit. Dissociation maximale = 100%.

4. **Prédire/agir mesuré** (E4) — dissociation **100%** en BattleSexes/Dilemme/Chicken (le joueur prédit CD/DC et joue CC), **0%** en StagHunt/Harmony/Coordination (les deux Nash symétriques, la prédiction et l'action coïncident par construction). C'est le pattern empirique du papier : le LLM **sait** mais **n'agit pas** sur les chambres à conflit.


## 8. Exercices

Cette section rassemble **3 exercices progressifs** sur la dissociation prédire/agir.

### Exercice 1 — Remplacer le joueur simulé par un vrai LLM (openai-compatible)

L'objectif : remplacer `sticky_preferred` par un appel provider réel. Quand `OPENAI_API_KEY` (ou équivalent) est présent dans `os.environ`, le notebook appelle le modèle ; sinon, il reste sur le stub `sticky_preferred`. C'est la cellule-type d'extension **RECOVERABLE-LOCAL** (cf [sota-not-workaround.md](../../.claude/rules/sota-not-workaround.md)).

### Exercice 2 — Caractériser les swaps qui augmentent la dissociation

L'objectif : pour chaque chambre X, lister **tous** les swaps qui transforment le Nash et mesurer la dissociation sticky vs BR sur chaque swap. Identifier les swaps qui **cachent** la dissociation (BattleSexes+C12) vs ceux qui la **révèlent** (Dilemme+C23).

### Exercice 3 — Cyclicité de BattleSexes : formalisation en logique modale

L'objectif : proposer un encodage **non-transitif** de BattleSexes qui préserve ses deux Nash (C,D) et (D,C). Indice : utiliser des préférences **lexicographiques** ou une **logique modale KD45**.

In [8]:
# Exercice 1 : Remplacer le joueur simule par un vrai LLM (openai-compatible)
def call_llm_provider(history: List[Tuple[str, str]], player: str,
                      api_key: str = None) -> str:
    """
    Appelle un modele openai-compatible (OpenAI, OpenRouter, Anthropic via gateway).
    Si pas de cle API, retourne un stub C.1 (None) -- le notebook reste executable.

    Indice etudiant :
      - Utiliser `os.environ.get("OPENAI_API_KEY")` ou equivalent
      - Prompt : convertir l'historique en regles textuelles (F/J) comme dans le papier
      - Temperature 0, max_tokens 1 (mono-token)
      - Si le provider repond, retourner 'C' ou 'D' ; sinon retourner None (stub)
    """
    api_key = api_key or os.environ.get("OPENAI_API_KEY")
    if not api_key:
        # Pas de provider : stub C.1
        return None
    # TODO etudiant : implementer l'appel provider
    # Indice : openai.OpenAI(api_key=...).chat.completions.create(...)
    # Exemple : client = openai.OpenAI(api_key=api_key)
    #           resp = client.chat.completions.create(
    #               model="gpt-4",
    #               messages=[{"role": "user", "content": build_prompt(history, player)}],
    #               temperature=0, max_tokens=1)
    #           return parse_action(resp.choices[1.message.content)
    return None  # Stub C.1


# Indice : pour la construction du prompt, transformer l'historique en texte :
def build_prompt_template(history: List[Tuple[str, str]], player: str, game_name: str) -> str:
    """Template du prompt envoye au modele. Les regles textuelles F/J evitement le biais semantique."""
    opponent = "Col" if player == "Row" else "Row"
    rows = [
        f"You are playing a repeated {game_name} game.",
        f"On each round, you choose F or J. The other player ({opponent}) also chooses.",
        f"History (most recent last):",
    ]
    for i, (r, c) in enumerate(history):
        rows.append(f"  Round {i+1}: F" if (r == "C" if player == "Row" else c == "C") else f"  Round {i+1}: J")
    rows.append("Choose F or J for the next round. Reply with one character only.")
    return "\n".join(rows)


# Stub : si pas de cle, on retourne None et le notebook reste executable
print("=== Exercice 1 : call_llm_provider ===")
result = call_llm_provider([("C", "C"), ("C", "C")], "Row")
print(f"Sans cle API, retourne : {result}")
print("=> Pour utiliser un vrai LLM : configurer OPENAI_API_KEY dans .env ou os.environ")

=== Exercice 1 : call_llm_provider ===
Sans cle API, retourne : None
=> Pour utiliser un vrai LLM : configurer OPENAI_API_KEY dans .env ou os.environ


In [9]:
# Exercice 2 : Caracteriser les swaps qui augmenent la dissociation
def dissociation_post_swap(g: OrdinalGame, swap: str,
                          n_total: int = 20, swap_round: int = 10) -> Tuple[float, float]:
    """
    Mesure la dissociation sticky vs BR apres le swap.
    Retourne (dissociation_sticky_post, dissociation_BR_post) en pourcentage.
    """
    h_sticky = play_with_swap(g, swap, swap_round=swap_round, n_total=n_total, mode="sticky_preferred")
    h_br = play_with_swap(g, swap, swap_round=swap_round, n_total=n_total, mode="best_response")
    d_sticky = dissociation_rate(g, h_sticky[swap_round:])  # post-swap seulement
    d_br = dissociation_rate(g, h_br[swap_round:])
    return d_sticky, d_br


# Indice etudiant : pour chaque chambre X, iterer sur tous les swaps R{i}{j} et C{i}{j}
# valides (0 <= i < j <= 3), mesurer dissociation_post_swap(X, swap), et retourner
# les swaps qui maximisent la dissociation sticky vs BR.
def find_max_dissociation_swap(g: OrdinalGame) -> List[Tuple[str, float, float]]:
    """
    Pour la chambre g, retourne les swaps (swap, d_sticky, d_br) tries par
    dissociation sticky decroissante.
    """
    # TODO etudiant : iterer sur tous les swaps valides (6 R-swaps + 6 C-swaps),
    # appeler dissociation_post_swap, et retourner la liste triee.
    return []  # Stub C.1


# Demonstration partielle : sur Dilemme
print("=== Exercice 2 : swaps qui revelent la dissociation sur Dilemme ===")
results_dilemme = []
for i in range(4):
    for j in range(i+1, 4):
        for kind in ["R", "C"]:
            swap = f"{kind}{i}{j}"
            d_s, d_b = dissociation_post_swap(g=CLASSIC_GAMES["Dilemme"], swap=swap)
            results_dilemme.append((swap, d_s, d_b))
# Trier par dissociation sticky decroissante
results_dilemme.sort(key=lambda x: -x[1])
print(f"{'Swap':6s} {'d_sticky_post':15s} {'d_BR_post':15s}")
for swap, d_s, d_b in results_dilemme[:6]:
    print(f"{swap:6s} {d_s:>13.0%}  {d_b:>13.0%}")

=== Exercice 2 : swaps qui revelent la dissociation sur Dilemme ===
Swap   d_sticky_post   d_BR_post      
R01             100%            50%
C01             100%             0%
R02             100%             0%
C02             100%            50%
R03             100%             0%
C03             100%             0%


In [10]:
# Exercice 3 : Cyclicite de BattleSexes -- formalisation non-transitive
# Indice : pour representer BoS canonique avec 2 Nash (C,D) et (D,C), il faut
# autoriser des preferences NON transitives (le joueur peut preferer C a D,
# D a (C,C), et (C,C) a C, etc.). Une solution : utiliser des "circles de
# preference" plutot que des rangs lineaires.

from typing import Dict, Set  # noqa: E402  (import local pour la cellule exercice)

def best_response_nontransitive(g_cyclic: Dict[Tuple[str, str], Set[str]],
                                player: str, opponent_action: str) -> str:
    """
    Pour une representation cyclique des preferences :
    g_cyclic[action] = ensemble des actions strictement preferees.
    Retourne la meilleure reponse selon cette relation cyclique.

    Indice etudiant :
      - Si g_cyclic[(C,C)] contient D, alors C < D quand (C,C) est joue
      - Si g_cyclic[(D,C)] contient C, alors D < C quand (D,C) est joue
      - Pour BattleSexes : definir les 4 ensembles cycliques
    """
    # TODO etudiant : definir les 4 ensembles cycliques pour BattleSexes
    # et implementer la selection d'action
    return "C"  # Stub C.1


# Demonstration : pour BattleSexes canonique, une representation cyclique
# pourrait etre :
# - En (C,C) : Row prefere C (relation C < D, i.e. D prefere)
# - En (C,D) : Row prefere D (relation C > D, i.e. C prefere encore)
# - En (D,C) : Row prefere C (relation D < C)
# - En (D,D) : Row prefere C (relation D > C)
# Cette relation est CYCLIQUE : C < D < C, violant la transitivite.
print("=== Exercice 3 : cyclicite de BattleSexes ===")
print("Representation cyclique possible :")
print("  (C,C) : Row prefere C | (C,D) : Row prefere D")
print("  (D,C) : Row prefere C | (D,D) : Row prefere C")
print()
print("Cycle : C < D (en CC) -> D < C (en CD) -> C < D (en DC) -> D < C (en DD)")
print("=> Non representable en ordinal strict transitif.")
print("=> Stub : completer best_response_nontransitive avec les 4 ensembles.")

=== Exercice 3 : cyclicite de BattleSexes ===
Representation cyclique possible :
  (C,C) : Row prefere C | (C,D) : Row prefere D
  (D,C) : Row prefere C | (D,D) : Row prefere C

Cycle : C < D (en CC) -> D < C (en CD) -> C < D (en DC) -> D < C (en DD)
=> Non representable en ordinal strict transitif.
=> Stub : completer best_response_nontransitive avec les 4 ensembles.


## 9. Conclusion

Le joueur LLM (modélisé en `sticky_preferred`) performe **structurellement** différemment selon la chambre Robinson-Goforth. En particulier :

- **BattleSexes, Dilemme, Chicken** : dissociation **maximale** — le joueur colle à son option préférée et **n'atteint pas** le Nash. C'est exactement le pattern empirique du papier (Mei et al. 2025).

- **StagHunt, Harmony, Coordination** : dissociation **nulle** — l'option préférée (C,C) coïncide avec le Nash. Le joueur performe "bien" par accident.

L'apport **conceptuel** de ce notebook est de montrer que la dissociation (b) du papier — **GPT-4 prédit l'alternance et n'agit pas** — est une **propriété structurelle** des chambres à conflit (BoS, Chicken), pas un artefact du modèle. Et la grammaire R-G (swaps en cours de partie) permet de **révéler** la dissociation quand elle est accidentellement cachée.

**Substrat EPITA** (#12254) : les personas de `2025-Epita-Intelligence-Symbolique` permettent l'extension directe : l'agent qui anticipe le contre-argument le réfute-t-il ? Plusieurs agents en désaccord sur la méta-action ? Ces questions héritent la grammaire de dissociation et la mesure explicite.

## Sources

- Mei et al., *Playing Repeated Games with Large Language Models*, Nature Human Behaviour (2025), [s41562-025-02172-y](https://www.nature.com/articles/s41562-025-02172-y) — page lue 2026-08-22.
- Robinson & Goforth, *The Topology of the 2x2 Games* (2005) — implémentation dans `GameTheory-3` cellule 5 (`OrdinalGame`).
- Bruns, *Austausch und Gerechtigkeit* (1975) — notion de transmutation (information nouvelle vs déplacement), voir aussi GameTheory-21 (Loi III, transformations vs morphismes).
- GameTheory-3 (chambres R-G) : [GameTheory-3-Topology2x2.ipynb](GameTheory-3-Topology2x2.ipynb)
- GameTheory-21 (morphisme fini, swaps préservants) : [GameTheory-21-Deux-Especes-de-Fleches.ipynb](GameTheory-21-Deux-Especes-de-Fleches.ipynb)

***

**Navigation** : [GameTheory-3](GameTheory-3-Topology2x2.ipynb) · **GameTheory-3c-Le-Joueur-LLM** · [GameTheory-21](GameTheory-21-Deux-Especes-de-Fleches.ipynb)